# Notebook 02: Cross-Domain Analysis

This notebook demonstrates the **domain shift evaluation pipeline** from the
`ragweed_toolkit.embeddings` module (Chapter 6, Act II).

When a detection model trained on one domain (e.g., Chilean field images) is
deployed to a different domain (e.g., European datasets), performance can
collapse. The toolkit provides tools to:

1. **Extract ResNet-50 embeddings** (2048-D feature vectors) from images
2. **Compute Maximum Mean Discrepancy (MMD)** between training and target domains
3. **Run a permutation test** for statistical significance
4. **Apply the deployment gate** to get an actionable risk tier
5. **Visualize** domain separation with UMAP and MMD heatmaps

All data in this notebook is synthetic -- no images or GPU are needed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

## 1. Generate Synthetic Embeddings

In production, a `FeatureExtractor` extracts 2048-D ResNet-50 features from
images. Here we simulate three domains with controlled distributional shifts:

- **Chile (training)**: The reference distribution
- **Spain**: Similar imaging conditions (small shift)
- **Germany**: Different climate, vegetation, cameras (large shift)

In [ ]:
d = 128  # reduced dim for speed; real pipeline uses 2048

# Chile (training domain) -- centered at origin
n_chile = 200
X_chile = np.random.randn(n_chile, d) * 1.0

# Spain -- small shift (similar climate/cameras)
n_spain = 150
X_spain = np.random.randn(n_spain, d) * 1.0 + 0.15

# Germany -- large shift (different climate, vegetation, cameras)
n_germany = 120
X_germany = np.random.randn(n_germany, d) * 1.5 + 1.5

domains = {
    "Chile (train)": X_chile,
    "Spain": X_spain,
    "Germany": X_germany,
}

print("=== Synthetic Embedding Domains ===")
for name, X in domains.items():
    print(f"  {name:20s}: {X.shape[0]:4d} samples, {X.shape[1]}D, "
          f"mean={X.mean():.2f}, std={X.std():.2f}")

## 2. Compute Pairwise MMD Matrix

The MMD (Maximum Mean Discrepancy) measures the distance between two
distributions in a reproducing kernel Hilbert space. Higher MMD means
more domain shift, and higher risk of performance collapse.

In [ ]:
from ragweed_toolkit.embeddings import compute_mmd

domain_names = list(domains.keys())
domain_arrays = list(domains.values())
n_domains = len(domain_names)

# Compute pairwise MMD matrix
mmd_matrix = np.zeros((n_domains, n_domains))
for i in range(n_domains):
    for j in range(i + 1, n_domains):
        mmd_val = compute_mmd(domain_arrays[i], domain_arrays[j])
        mmd_matrix[i, j] = mmd_val
        mmd_matrix[j, i] = mmd_val

print("=== Pairwise MMD Matrix ===")
mmd_df = pd.DataFrame(mmd_matrix, index=domain_names, columns=domain_names)
print(mmd_df.round(3).to_string())

## 3. Deployment Gate

The deployment gate maps MMD values to four actionable tiers:

| MMD Range | Tier | Recommendation |
|-----------|------|----------------|
| < 0.15 | Green | Deploy directly |
| 0.15 -- 0.30 | Yellow | Deploy with augmentation |
| 0.30 -- 0.45 | Orange | Active learning required |
| > 0.45 | Red | Collect new training data |

In [ ]:
from ragweed_toolkit.embeddings import deployment_gate

print("=== Deployment Gate Results ===")
print(f"{'Target':<20} {'MMD':>8} {'Tier':<10} {'Recommendation'}")
print("-" * 80)

for name, X_target in [("Spain", X_spain), ("Germany", X_germany)]:
    mmd_val = compute_mmd(X_chile, X_target)
    result = deployment_gate(mmd_val)
    print(f"{name:<20} {mmd_val:>8.3f} {result.tier:<10} {result.recommendation}")

## 4. Permutation Test

The permutation test assesses whether the observed MMD is statistically
significant. Under H0 (identical distributions), the domain labels are
randomly shuffled. A p-value < 0.05 means the domain shift is real.

In [ ]:
from ragweed_toolkit.embeddings import permutation_test

print("=== Permutation Tests (200 permutations for speed) ===")
for name, X_target in [("Spain", X_spain), ("Germany", X_germany)]:
    mmd_val, p_value = permutation_test(
        X_chile, X_target,
        n_permutations=200,
        random_state=42,
    )
    print(f"\n  Chile vs {name}:")
    print(f"    MMD       = {mmd_val:.4f}")
    print(f"    p-value   = {p_value:.4f}")
    print(f"    Reject H0 = {p_value < 0.05} (alpha=0.05)")

## 5. UMAP Visualization

Reduce the combined embeddings to 2D with UMAP and color by domain.

In [ ]:
from ragweed_toolkit.embeddings import reduce_embeddings

# Combine all embeddings with domain labels
all_embeddings = np.vstack([X_chile, X_spain, X_germany])
domain_labels = (
    ["Chile (train)"] * n_chile
    + ["Spain"] * n_spain
    + ["Germany"] * n_germany
)

# Reduce with PCA (fast, no umap-learn needed)
# For UMAP, change method="umap" (requires: pip install umap-learn)
df_reduced = reduce_embeddings(
    all_embeddings,
    method="pca",
    n_components=2,
    metadata={"domain": domain_labels},
)

print(f"Reduced {all_embeddings.shape} -> {df_reduced.shape}")
df_reduced.head()

In [ ]:
from ragweed_toolkit.viz.style import set_publication_style

set_publication_style()

domain_colors = {
    "Chile (train)": "#2ecc71",
    "Spain": "#f1c40f",
    "Germany": "#e74c3c",
}

fig, ax = plt.subplots(figsize=(9, 7))

for domain_name, color in domain_colors.items():
    mask = df_reduced["domain"] == domain_name
    subset = df_reduced[mask]
    ax.scatter(
        subset["pca_x"], subset["pca_y"],
        c=color, label=domain_name,
        s=25, alpha=0.7, edgecolors="black", linewidths=0.3,
    )

ax.set_xlabel("PCA Component 1")
ax.set_ylabel("PCA Component 2")
ax.set_title("Domain Separation (PCA Projection)", fontweight="bold")
ax.legend(title="Domain")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 6. MMD Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(mmd_matrix, cmap="YlOrRd", vmin=0)
ax.set_xticks(range(n_domains))
ax.set_yticks(range(n_domains))
ax.set_xticklabels(domain_names, rotation=30, ha="right")
ax.set_yticklabels(domain_names)

# Annotate cells
for i in range(n_domains):
    for j in range(n_domains):
        color = "white" if mmd_matrix[i, j] > 0.3 else "black"
        ax.text(j, i, f"{mmd_matrix[i, j]:.3f}",
                ha="center", va="center", fontsize=11, color=color)

plt.colorbar(im, ax=ax, label="MMD", shrink=0.8)
ax.set_title("Pairwise MMD Matrix", fontweight="bold")
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated the cross-domain evaluation pipeline:

- **MMD** quantifies distributional distance between domains.
- **Permutation tests** provide p-values for statistical significance.
- The **deployment gate** maps MMD to four risk tiers with clear recommendations.
- **PCA/UMAP** visualizations reveal domain clusters and overlap.

### Implications for Deployment

- A model trained on Chilean data may deploy directly to similar Mediterranean
  regions (green gate) but will likely fail on Central European data (red gate).
- The chapter showed that **multi-domain training** (combining Chile + Europe)
  recovers mAP50 from 0.108 to 0.874 (Section 6.3).

### Next Steps

- **Notebook 03**: Use the detection output as input for geostatistical mapping
- **Notebook 04**: Correlate satellite indices with weed density